In [ ]:
# ==============================================================================
# CLOTHMATICS 3D GHOST MANNEQUIN - OUTFIT ROUTER (v9.3.0)
# ==============================================================================
# Architecture:
# 1. 🛡️ Safe Baseline: Non-destructive raw FLUX generation, high-speed FP16 compute.
# 2. 🎨 Color-Managed Input Decoding: sRGB ICC profile normalization via ImageCms,
#    EXIF transposition, and alpha channel preservation onto pure white (#FFFFFF).
# 3. 📝 Preservation-First Prompting: Subordinate category & observation hints.
# 4. 🚀 Warm Dual-GPU (Qwen3 on GPU 0, FLUX.2 Transformer & FP32 Tiled VAE on GPU 1).
# 5. 📡 Persistent Keep-Alive Server Loop with Real-Time Request Streaming.
# ==============================================================================

import os, sys, json, time, re, socket, subprocess, textwrap, urllib.request, shutil
from pathlib import Path

print("Starting ClothMatics outfit router (v9.3.0; v9.2 renderer unchanged)...\n")

def read_kaggle_secret(name):
    value = os.environ.get(name, '').strip()
    if not value:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(name).strip()
        except Exception:
            pass
    if not value:
        raise RuntimeError(f'Add {name} to Kaggle Secrets and enable it for this notebook before running.')
    return value

# Validate configuration before downloading weights or taking GPU memory.
SYNC_TOKEN = read_kaggle_secret('CLOTHMATICS_SYNC_TOKEN')

# ------------------------------------------------------------------------------
# STEP 1: Clean Startup (Terminate any old server, tunnel, or file handles)
# ------------------------------------------------------------------------------
for fh_name in ['API_LOG_FILE', 'TUNNEL_LOG_FILE']:
    if fh_name in globals():
        try:
            globals()[fh_name].close()
        except Exception:
            pass

for proc_name in ['API_PROCESS', 'TUNNEL_PROCESS']:
    if proc_name in globals() and globals()[proc_name].poll() is None:
        try:
            print(f"Stopping previous {proc_name}...")
            globals()[proc_name].terminate()
            globals()[proc_name].wait(timeout=3)
        except Exception:
            pass

# Stop only subprocesses created by this notebook. Do not kill other notebooks.

# ------------------------------------------------------------------------------
# STEP 2: Dedicated Project Environment & Dependencies
# ------------------------------------------------------------------------------
PROJECT = Path('/kaggle/working/clothmatics_ghost_env')
PROJECT.mkdir(parents=True, exist_ok=True)
ENV = PROJECT / '.venv'
PYTHON = sys.executable

# Self-healing environment creation: avoid ensurepip crashes on Debian/Ubuntu
try:
    import venv
    if not ENV.exists():
        venv.EnvBuilder(with_pip=False, system_site_packages=True).create(ENV)

    cand = str(ENV / 'bin' / 'python')
    if (ENV / 'bin' / 'python').exists():
        probe = subprocess.run([cand, '-c', 'import sys; print(sys.version)'], capture_output=True)
        if probe.returncode == 0:
            PYTHON = cand
except Exception as e:
    print(f"Note on environment ({e}), using Kaggle runtime Python directly.")
    PYTHON = sys.executable

PKGS = [
    "fastapi", "uvicorn[standard]", "python-multipart", "opencv-python-headless",
    "diffusers==0.40.0", "transformers==5.16.1", "accelerate==1.14.0",
    "bitsandbytes==0.50.2", "peft==0.20.0", "safetensors==0.8.0",
    "huggingface-hub", "Pillow>=10.4.0", "scipy==1.14.1",
    "google-cloud-bigquery-storage>=2.0.0"
]

print("Installing & verifying required packages...")
pip_flags = ["install", "-q", "--no-warn-conflicts"]
install_success = False
if PYTHON != sys.executable:
    try:
        subprocess.run([sys.executable, "-m", "pip", "--python", PYTHON] + pip_flags + PKGS, check=True)
        install_success = True
    except Exception:
        print("Venv install failed, switching to Kaggle runtime Python...")
        PYTHON = sys.executable

if not install_success:
    subprocess.run([sys.executable, "-m", "pip"] + pip_flags + PKGS, check=True)

# GPU Hardware Confirmation
probe = textwrap.dedent("""
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Kaggle GPU not active! Right sidebar me Accelerator -> GPU T4 x2 select karein.')
    count = torch.cuda.device_count()
    print(f"Detected {count} GPU(s):")
    for i in range(count):
        free, total = torch.cuda.mem_get_info(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({round(free/2**30, 2)} GB / {round(total/2**30, 2)} GB free)")
""")
subprocess.run([PYTHON, "-u", "-c", probe], check=True)
print("✅ GPU hardware confirmed.")

# ------------------------------------------------------------------------------
# STEP 3: Pre-cache FLUX.2-klein-4B & Qwen3 Model Weights
# ------------------------------------------------------------------------------
print("\n" + "="*80)
print("📥 STEP 3: Downloading & Pre-caching FLUX.2-klein-4B Weights to Local SSD...")
print("="*80)
cache_probe = textwrap.dedent("""
    import os, sys
    from huggingface_hub import snapshot_download

    artifact_id = "black-forest-labs/FLUX.2-klein-4B"
    revision = "e7b7dc27f91deacad38e78976d1f2b499d76a294"
    print(f"Downloading/verifying weights for {artifact_id}...")
    print(f"Using 8 parallel download workers (progress bar displayed below):")
    path = snapshot_download(
        repo_id=artifact_id,
        revision=revision,
        ignore_patterns=["*.msgpack", "*.onnx"],
        max_workers=8
    )
    print(f"✅ All model weights ready in local SSD cache!")
""")
subprocess.run([PYTHON, "-u", "-c", cache_probe], check=True)

parser_cache_probe = textwrap.dedent("""
    from huggingface_hub import snapshot_download
    artifact_id = "mattmdjaga/segformer_b2_clothes"
    revision = "584abc1e1d260e23c0fc627c5217a09b2b461046"
    print(f"Downloading/verifying CPU outfit parser {artifact_id}...")
    snapshot_download(repo_id=artifact_id, revision=revision, max_workers=8)
    print("✅ Outfit parser weights ready in local SSD cache!")
""")
subprocess.run([PYTHON, "-u", "-c", parser_cache_probe], check=True)

# ------------------------------------------------------------------------------
# STEP 4: High-Speed Warm In-Memory FastAPI Server + Color Calibration
# ------------------------------------------------------------------------------
WARM_SERVER_CODE = 'import io, json, os, subprocess, sys, tempfile, threading, time, uuid, re, warnings\nimport importlib.metadata\nimport asyncio, hashlib\nfrom collections import OrderedDict\nfrom prompt_contract import normalize_category, normalize_manifest, pack_prompt, category_dimensions, CONTRACT_VERSION\nfrom dataclasses import dataclass\nfrom typing import Optional, Dict, Any, Tuple, List\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nfrom PIL import Image, ImageOps, ImageCms, UnidentifiedImageError\nimport torch\nfrom fastapi import FastAPI, File, Form, HTTPException, UploadFile, Request\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import Response\n\n# ==============================================================================\n# 1. CONSTANTS, PROFILES & COLOR MANAGEMENT\n# ==============================================================================\nPIPELINE_VERSION = "9.2.0-multicolor"\nMAX_UPLOAD_BYTES = 20 * 1024 * 1024  # 20 MB\nMAX_INPUT_PIXELS = 24_000_000        # 24 Mpx\nSRGB_PROFILE = ImageCms.ImageCmsProfile(ImageCms.createProfile("sRGB"))\nSRGB_ICC = SRGB_PROFILE.tobytes()\n\nclass InputImageError(ValueError):\n    """Raised when an uploaded input image violates size, mode or format constraints."""\n    pass\n\n@dataclass\nclass DecodedGarment:\n    rgb: Image.Image\n    alpha: Optional[Image.Image]\n    conditioning_rgb: Image.Image\n    notices: tuple\n\ndef decode_garment(data: bytes) -> DecodedGarment:\n    if not data or len(data) > MAX_UPLOAD_BYTES:\n        raise InputImageError("Image is empty or exceeds the upload limit (20MB)")\n    notices = []\n    try:\n        with warnings.catch_warnings():\n            warnings.simplefilter("error", Image.DecompressionBombWarning)\n            with Image.open(io.BytesIO(data)) as opened:\n                if getattr(opened, "n_frames", 1) != 1:\n                    raise InputImageError("Upload one still image")\n                if opened.width * opened.height > MAX_INPUT_PIXELS:\n                    raise InputImageError("Image exceeds the pixel limit (24M pixels)")\n                opened.load()\n                icc = opened.info.get("icc_profile")\n                oriented = ImageOps.exif_transpose(opened)\n                has_alpha = (\n                    "A" in oriented.getbands() or "transparency" in oriented.info\n                )\n                alpha = oriented.convert("RGBA").getchannel("A") if has_alpha else None\n                if alpha is not None:\n                    extrema = alpha.getextrema()\n                    if extrema[1] == 0:\n                        raise InputImageError("Image is fully transparent")\n                    if extrema == (255, 255):\n                        alpha = None\n                if oriented.mode == "LA":\n                    base = oriented.getchannel("L")\n                else:\n                    base = oriented if oriented.mode in ("RGB", "CMYK", "L", "LAB") else oriented.convert("RGB")\n                if icc:\n                    try:\n                        src_profile = ImageCms.ImageCmsProfile(io.BytesIO(icc))\n                        rgb = ImageCms.profileToProfile(\n                            base, src_profile, SRGB_PROFILE, outputMode="RGB"\n                        )\n                    except (ImageCms.PyCMSError, OSError, ValueError) as exc:\n                        raise InputImageError("Embedded color profile cannot be converted") from exc\n                else:\n                    if base.mode not in ("RGB", "L"):\n                        raise InputImageError("This color mode requires a valid embedded profile")\n                    rgb = base.convert("RGB")\n                    notices.append("untagged_input_assumed_srgb")\n                rgb = rgb.copy()\n                if alpha is not None:\n                    alpha = alpha.copy()\n                    rgba = rgb.convert("RGBA")\n                    rgba.putalpha(alpha)\n                    white = Image.new("RGBA", rgb.size, (255, 255, 255, 255))\n                    conditioning = Image.alpha_composite(white, rgba).convert("RGB")\n                else:\n                    conditioning = rgb.copy()\n                rgb.info.clear()\n                conditioning.info.clear()\n                return DecodedGarment(rgb, alpha, conditioning, tuple(notices))\n    except InputImageError:\n        raise\n    except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombWarning, getattr(Image, "DecompressionBombError", Exception)) as exc:\n        raise InputImageError("Unsupported or invalid image") from exc\n\ndef conditioning_thumbnail(decoded: DecodedGarment, max_size=(768, 1024)) -> Image.Image:\n    result = decoded.conditioning_rgb.copy()\n    result.thumbnail(max_size, Image.Resampling.LANCZOS)\n    return result\n\n# ==============================================================================\n# 2. CATEGORY NORMALIZATION & PRESERVATION PROMPTS\n# ==============================================================================\n# Category handling and token-aware prompts live in prompt_contract.py.\n\nclass WarmDualGpuEngine:\n    def __init__(self):\n        try:\n            from transformers import Qwen3ForCausalLM as TextEncoderModel\n        except (ImportError, AttributeError):\n            from transformers import AutoModelForCausalLM as TextEncoderModel\n        from transformers import BitsAndBytesConfig as TextQuant\n        from diffusers import Flux2KleinPipeline, Flux2Transformer2DModel, BitsAndBytesConfig as ImageQuant\n        \n        self.num_gpus = torch.cuda.device_count()\n        if self.num_gpus == 0:\n            raise RuntimeError(\'Enable a Kaggle GPU accelerator before starting the engine.\')\n        if self.num_gpus >= 2:\n            self.text_device = torch.device("cuda:0")\n            self.image_device = torch.device("cuda:1")\n            print("🚀 DUAL-GPU PIPELINE: GPU 0 (Qwen3 Text) | GPU 1 (FLUX Transformer)", flush=True)\n        elif self.num_gpus == 1:\n            self.text_device = torch.device("cuda:0")\n            self.image_device = torch.device("cuda:0")\n            print("⚡ SINGLE-GPU PIPELINE: GPU 0 (Shared)", flush=True)\n        else:\n            self.text_device = torch.device("cpu")\n            self.image_device = torch.device("cpu")\n\n        artifact_id = "black-forest-labs/FLUX.2-klein-4B"\n        self.revision = "e7b7dc27f91deacad38e78976d1f2b499d76a294"\n        self.compute_dtype = torch.float16\n\n        quant = dict(\n            load_in_4bit=True,\n            bnb_4bit_quant_type="nf4",\n            bnb_4bit_use_double_quant=True,\n            bnb_4bit_compute_dtype=self.compute_dtype\n        )\n        pinned = dict(revision=self.revision, trust_remote_code=False)\n\n        print(f"⚡ [1/2] Loading Qwen3 Text Encoder into {self.text_device} (compute: {self.compute_dtype})...", flush=True)\n        encoder = TextEncoderModel.from_pretrained(\n            artifact_id, subfolder="text_encoder",\n            quantization_config=TextQuant(**quant), torch_dtype=self.compute_dtype,\n            device_map={\'\': str(self.text_device)} if torch.cuda.is_available() else None, **pinned\n        )\n        self.text_pipe = Flux2KleinPipeline.from_pretrained(\n            artifact_id, text_encoder=encoder, transformer=None,\n            vae=None, torch_dtype=self.compute_dtype, **pinned\n        )\n\n        print(f"⚡ [2/2] Loading FLUX.2 Transformer into {self.image_device}...", flush=True)\n        transformer = Flux2Transformer2DModel.from_pretrained(\n            artifact_id, subfolder="transformer",\n            quantization_config=ImageQuant(**quant), torch_dtype=self.compute_dtype,\n            device_map={\'\': str(self.image_device)} if torch.cuda.is_available() else None, **pinned\n        )\n\n        print(f"⚡ [3/3] Assembling FLUX.2 Pipeline & VAE into {self.image_device}...", flush=True)\n        self.image_pipe = Flux2KleinPipeline.from_pretrained(\n            artifact_id, transformer=transformer,\n            text_encoder=None, tokenizer=None, torch_dtype=self.compute_dtype, **pinned\n        )\n        self.image_pipe.vae.to(device=self.image_device, dtype=torch.float32)\n        if hasattr(self.image_pipe.vae, \'enable_tiling\'):\n            try:\n                self.image_pipe.vae.enable_tiling()\n            except Exception:\n                pass\n\n        # VAE Precision Bridge: automatically cast incoming latents to VAE\'s FP32 precision\n        target_vae_dev = self.image_device\n        target_vae_dtype = torch.float32\n\n        orig_vae_decode = self.image_pipe.vae.decode\n        def safe_vae_decode(latents, *args, **kwargs):\n            if torch.is_tensor(latents):\n                latents = latents.to(device=target_vae_dev, dtype=target_vae_dtype)\n            return orig_vae_decode(latents, *args, **kwargs)\n        self.image_pipe.vae.decode = safe_vae_decode\n\n        if hasattr(self.image_pipe.vae, \'_decode\'):\n            orig_vae_internal_decode = self.image_pipe.vae._decode\n            def safe_vae_internal_decode(z, *args, **kwargs):\n                if torch.is_tensor(z):\n                    z = z.to(device=target_vae_dev, dtype=target_vae_dtype)\n                return orig_vae_internal_decode(z, *args, **kwargs)\n            self.image_pipe.vae._decode = safe_vae_internal_decode\n\n        if hasattr(self.image_pipe.vae, \'encode\'):\n            orig_vae_encode = self.image_pipe.vae.encode\n            def safe_vae_encode(x, *args, **kwargs):\n                if torch.is_tensor(x):\n                    x = x.to(device=target_vae_dev, dtype=target_vae_dtype)\n                return orig_vae_encode(x, *args, **kwargs)\n            self.image_pipe.vae.encode = safe_vae_encode\n\n        vae_scale = 8\n        if hasattr(self.image_pipe, \'vae_scale_factor\'):\n            vae_scale = self.image_pipe.vae_scale_factor\n\n        pkg_versions = {}\n        for pkg in (\'torch\', \'diffusers\', \'transformers\', \'accelerate\', \'bitsandbytes\', \'peft\'):\n            try:\n                pkg_versions[pkg] = importlib.metadata.version(pkg)\n            except Exception:\n                pkg_versions[pkg] = "unknown"\n\n        self.metadata = {\n            "model_id": artifact_id,\n            "revision": self.revision,\n            "is_distilled": True,\n            "pipeline_class": "Flux2KleinPipeline",\n            "text_device": str(self.text_device),\n            "image_device": str(self.image_device),\n            "compute_dtype": str(self.compute_dtype),\n            "vae_scale_factor": vae_scale,\n            "postprocess_mode": "safe_baseline",\n            "destructive_postprocessing": False,\n            "packages": pkg_versions,\n        }\n        print(f"📊 Engine Metadata: {json.dumps(self.metadata)}", flush=True)\n        self.is_warm = False\n        print(f"✅ Models loaded in VRAM (GPU 0: Text, GPU 1: Transformer & VAE). Ready for preflight warm-up pass.", flush=True)\n\n    def generate(\n        self,\n        source_img: Optional[Image.Image] = None,\n        category: str = "garment",\n        custom_prompt: Optional[str] = None,\n        details: Optional[str] = None,\n        seed: int = 42,\n        width: Optional[int] = None,\n        height: Optional[int] = None,\n        quality: str = "standard",\n        decoded: Optional[DecodedGarment] = None,\n        manifest: Optional[dict] = None,\n    ) -> tuple:\n        timings = {}\n        t_all_start = time.perf_counter()\n        cat_norm = normalize_category(category)\n\n        if decoded is None:\n            if source_img is None:\n                raise ValueError("Either decoded or source_img must be provided")\n            rgb = source_img.convert("RGB")\n            conditioning = rgb.copy()\n            decoded = DecodedGarment(rgb=rgb, alpha=None, conditioning_rgb=conditioning, notices=("synthetic_warmup",))\n\n        if width is None or height is None:\n            width, height = category_dimensions(cat_norm, decoded.conditioning_rgb.size, quality)\n\n        # Warmup is the only call allowed without a real appearance manifest.\n        if manifest is None and \'synthetic_warmup\' in decoded.notices:\n            manifest = {\'category\':cat_norm,\'colorAndFinish\':\'Keep the reference grey\',\'surfaceTextureAndWeave\':\'Smooth fabric\'}\n        tokenizer = self.text_pipe.tokenizer\n        prompt, prompt_report = pack_prompt(cat_norm, manifest, tokenizer)\n\n        # 1. Text encoding on GPU 0\n        t0 = time.perf_counter()\n        if torch.cuda.is_available() and self.text_device.type == "cuda":\n            torch.cuda.synchronize(self.text_device)\n        with torch.inference_mode():\n            embeds, _ = self.text_pipe.encode_prompt(\n                prompt=prompt, device=self.text_device, max_sequence_length=512\n            )\n        if torch.cuda.is_available() and self.text_device.type == "cuda":\n            torch.cuda.synchronize(self.text_device)\n        timings[\'text_enc\'] = round((time.perf_counter() - t0) * 1000, 1)\n\n        # 2. Embeddings transfer (GPU 0 -> GPU 1)\n        t0 = time.perf_counter()\n        embeds = embeds.to(device=self.image_device, dtype=self.compute_dtype)\n        if torch.cuda.is_available() and self.image_device.type == "cuda":\n            torch.cuda.synchronize(self.image_device)\n        timings[\'embed_transfer\'] = round((time.perf_counter() - t0) * 1000, 1)\n\n        # 3. Conditioning Image Preparation\n        t0 = time.perf_counter()\n        conditioned = conditioning_thumbnail(decoded, max_size=(width, height))\n        timings[\'preprocess\'] = round((time.perf_counter() - t0) * 1000, 1)\n\n        # 4. Denoising Diffusion on GPU 1 (4 steps)\n        t0 = time.perf_counter()\n        with torch.inference_mode():\n            pipe_out = self.image_pipe(\n                image=conditioned,\n                prompt_embeds=embeds,\n                width=width,\n                height=height,\n                num_inference_steps=4,\n                guidance_scale=1.0,\n                generator=torch.Generator(device="cpu").manual_seed(seed),\n            )\n            raw_out = pipe_out.images[0]\n        if torch.cuda.is_available() and self.image_device.type == "cuda":\n            torch.cuda.synchronize(self.image_device)\n        timings[\'denoise\'] = round((time.perf_counter() - t0) * 1000, 1)\n\n        # 5. Output to PIL Image\n        t0 = time.perf_counter()\n        if isinstance(raw_out, Image.Image):\n            gen_pil = raw_out\n        else:\n            arr = np.array(raw_out)\n            if arr.dtype != np.uint8:\n                arr = np.rint(np.clip(arr * 255.0, 0, 255)).astype(np.uint8)\n            gen_pil = Image.fromarray(arr)\n        timings[\'decode\'] = round((time.perf_counter() - t0) * 1000, 1)\n\n        final_pil = gen_pil.copy()\n        timings[\'color\'] = 0.0\n        timings[\'prune\'] = 0.0\n        timings[\'repair\'] = 0.0\n\n        report = {\n            "quality_status": "unverified",\n            "postprocess_reason": "safe_baseline_no_destructive_postprocessing",\n            "postprocess_mode": "safe_baseline",\n            "category": cat_norm,\n            "width": width,\n            "height": height,\n            "seed": seed,\n            "model_revision": self.revision,\n            "notices": list(decoded.notices),\n            "prompt_report": prompt_report,\n            "timings_ms": timings\n        }\n\n        timings[\'total\'] = round((time.perf_counter() - t_all_start) * 1000, 1)\n        return final_pil, timings, report\n\n# ==============================================================================\n# 4. FASTAPI APPLICATION & BOUNDED QUEUE\n# ==============================================================================\napp = FastAPI(title="ClothMatics Safe Baseline Ghost Mannequin Engine")\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_credentials=True,\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\nENGINE = None\nENGINE_STATUS = "initializing"\nENGINE_ERROR = None\nLOCK = threading.Lock()\n\ndef load_engine_worker():\n    global ENGINE, ENGINE_STATUS, ENGINE_ERROR\n    try:\n        ENGINE_STATUS = "loading_models"\n        print("⚡ Loading Dual-GPU Models into VRAM...", flush=True)\n        engine = WarmDualGpuEngine()\n        ENGINE_STATUS = "preflight_inference"\n        print("🔥 Executing representative warm-up preflight inference...", flush=True)\n        dummy = Image.new("RGB", (576, 768), (245, 245, 245))\n        _, warm_timings, _ = engine.generate(source_img=dummy, category="shirt", width=576, height=768)\n        engine.is_warm = True\n        ENGINE = engine\n        ENGINE_STATUS = "ready"\n        print(f"🎉 WARM-UP SUCCESSFUL! Latency: {warm_timings[\'total\']}ms. Permanent VRAM readiness active.", flush=True)\n    except Exception as e:\n        import traceback\n        err_msg = traceback.format_exc()\n        ENGINE_ERROR = err_msg\n        ENGINE_STATUS = "error"\n        print(f"❌ Engine load error:\\n{err_msg}", flush=True)\n\n@app.on_event("startup")\ndef startup_load():\n    thread = threading.Thread(target=load_engine_worker, daemon=True)\n    thread.start()\n\n@app.get("/")\n@app.get("/health")\ndef health():\n    ready = ENGINE is not None and getattr(ENGINE, \'is_warm\', False)\n    gpu_info = []\n    if torch.cuda.is_available():\n        for i in range(torch.cuda.device_count()):\n            free, total = torch.cuda.mem_get_info(i)\n            gpu_info.append({\n                "gpu": i,\n                "name": torch.cuda.get_device_name(i),\n                "free_gb": round(free / 2**30, 2),\n                "total_gb": round(total / 2**30, 2)\n            })\n    return {\n        "status": "online" if ready else ENGINE_STATUS,\n        "service": "ClothMatics Safe Baseline Ghost Mannequin Engine",\n        "pipeline_version": PIPELINE_VERSION,\n        "ghost_contract_version": CONTRACT_VERSION,\n        "model": "FLUX.2-klein-4B NF4 (Warm Dual-GPU)",\n        "revision": getattr(ENGINE, \'revision\', \'e7b7dc27f91deacad38e78976d1f2b499d76a294\') if ENGINE else \'pending\',\n        "precision": "FP16 (NF4 Tensor Cores) + FP32 Tiled VAE",\n        "ready": ready,\n        "quality_status": "unverified",\n        "postprocess_mode": "safe_baseline",\n        "error": ENGINE_ERROR,\n        "gpus": gpu_info\n    }\n\n\n\n# This reviewed fragment is appended to ghost_server.py by the notebook builder.\n# Inference runs on a thread so /health and disconnect handling remain responsive.\nRESULT_CACHE = OrderedDict()\nCACHE_LOCK = threading.Lock()\nCACHE_TTL = 600\nCACHE_BYTES = 40 * 1024 * 1024\n\nfrom contextvars import ContextVar\nREQUEST_ID = ContextVar(\'request_id\', default=\'startup\')\n\ndef request_log(event, **details):\n    # Never log request headers, sync tokens, image bytes or raw user prompts.\n    print(json.dumps({\'event\':event, \'request_id\':REQUEST_ID.get(), **details}), flush=True)\n\n@app.middleware(\'http\')\nasync def log_request(request: Request, call_next):\n    if request.url.path in (\'/\', \'/health\'):\n        return await call_next(request)\n    context = REQUEST_ID.set(str(uuid.uuid4()))\n    started = time.perf_counter()\n    route = \'generate\' if request.url.path == \'/generate\' else (\'outfit\' if request.url.path == \'/outfit\' else \'other\')\n    request_log(\'request_received\', route=route)\n    try:\n        response = await call_next(request)\n        response.headers[\'X-Request-Id\'] = REQUEST_ID.get()\n        request_log(\'request_finished\', status=response.status_code, elapsed_ms=round((time.perf_counter()-started)*1000,1))\n        return response\n    except Exception as exc:\n        request_log(\'request_failed\', error_type=type(exc).__name__, elapsed_ms=round((time.perf_counter()-started)*1000,1))\n        raise\n    finally:\n        REQUEST_ID.reset(context)\n\ndef render_request(data, category, manifest, quality, seed):\n    key = hashlib.sha256(data + json.dumps([category, manifest, quality, seed], sort_keys=True).encode()).hexdigest()\n    with CACHE_LOCK:\n        now = time.monotonic()\n        for old_key, cached in list(RESULT_CACHE.items()):\n            if now - cached[0] > CACHE_TTL:\n                RESULT_CACHE.pop(old_key, None)\n        cached = RESULT_CACHE.get(key)\n        if cached:\n            request_log(\'cache_hit\', category=category, seed=seed)\n            return Response(cached[1], media_type=\'image/png\', headers={**cached[2], \'X-Ghost-Cache\':\'hit\'})\n    if not LOCK.acquire(blocking=False):\n        request_log(\'gpu_busy\', retry_after_seconds=15)\n        raise HTTPException(429, \'Another garment is being processed. Please retry.\', headers={\'Retry-After\':\'15\'})\n    try:\n        decoded = decode_garment(data)\n        if min(decoded.rgb.size) < 64:\n            raise InputImageError(\'The source garment image is too small\')\n        request_log(\'generation_started\', category=category, seed=seed, input_bytes=len(data), source_width=decoded.rgb.width, source_height=decoded.rgb.height, palette=manifest.get(\'palette\', []))\n        output, timings, report = ENGINE.generate(decoded=decoded, category=category, manifest=manifest, quality=quality, seed=seed)\n        buffer = io.BytesIO()\n        output.convert(\'RGB\').save(buffer, format=\'PNG\', icc_profile=SRGB_ICC)\n        payload = buffer.getvalue()\n        if not 500 <= len(payload) <= MAX_UPLOAD_BYTES:\n            raise RuntimeError(\'Generated output exceeds the supported size\')\n        headers = {\'X-Request-Id\':str(uuid.uuid4()), \'X-Pipeline-Version\':PIPELINE_VERSION,\n                   \'X-Ghost-Contract-Version\':str(CONTRACT_VERSION), \'X-Ghost-Seed\':str(seed), \'X-Ghost-Palette-Version\':\'1\',\n                   \'X-Quality-Status\':\'requires_visual_comparison\', \'X-Postprocess-Mode\':\'none\',\n                   \'X-Prompt-Tokens\':str(report[\'prompt_report\'][\'prompt_tokens\']),\n                   \'X-Generation-Time\':str(timings[\'total\']) + \'ms\'}\n        with CACHE_LOCK:\n            RESULT_CACHE[key] = (time.monotonic(), payload, headers)\n            while len(RESULT_CACHE) > 4 or sum(len(entry[1]) for entry in RESULT_CACHE.values()) > CACHE_BYTES:\n                RESULT_CACHE.popitem(last=False)\n        request_log(\'generation_finished\', category=category, seed=seed, output_width=report[\'width\'], output_height=report[\'height\'], prompt_report=report[\'prompt_report\'], timings_ms=timings, output_bytes=len(payload), quality_status=\'requires_visual_comparison\')\n        return Response(payload, media_type=\'image/png\', headers=headers)\n    except InputImageError as exc:\n        raise HTTPException(400, str(exc)) from exc\n    finally:\n        LOCK.release()\n\n@app.post(\'/generate\')\nasync def generate(image: UploadFile = File(...), category: str = Form(...),\n                   manifest: str = Form(...), contract_version: int = Form(...),\n                   quality: str = Form(\'high\'), seed: int = Form(42)):\n    try:\n        if ENGINE is None or not getattr(ENGINE, \'is_warm\', False):\n            raise HTTPException(503, \'Engine is warming up; retry shortly.\', headers={\'Retry-After\':\'15\'})\n        if contract_version != CONTRACT_VERSION:\n            raise HTTPException(409, \'Update the website to the appearance v2 contract.\')\n        if quality not in (\'standard\', \'high\') or not 0 <= seed < 2**32:\n            raise HTTPException(400, \'Unsupported quality or seed\')\n        if len(manifest) > 6000 or image.content_type not in (\'image/png\',\'image/jpeg\',\'image/webp\'):\n            raise HTTPException(400, \'Invalid garment manifest or image type\')\n        try:\n            category = normalize_category(category)\n            parsed = normalize_manifest(category, json.loads(manifest))\n        except (ValueError, TypeError) as exc:\n            raise HTTPException(400, \'Invalid or mismatched garment evidence\') from exc\n        data = await image.read(MAX_UPLOAD_BYTES + 1)\n        if not data or len(data) > MAX_UPLOAD_BYTES:\n            raise HTTPException(413, \'Empty image or upload exceeds 20 MB\')\n        # The thread owns the GPU lock even if an HTTP caller times out.\n        return await asyncio.to_thread(render_request, data, category, parsed, quality, seed)\n    finally:\n        await image.close()\n\n# Optional full-photo outfit route. Appended after request_handler.py by the\n# notebook builder; /generate remains untouched.\nimport base64\nfrom fastapi.responses import JSONResponse\nfrom outfit_parser import parse_outfit, OutfitParserError, PARSER_VERSION\n\nMAX_OUTFIT_ITEMS = 5\nMAX_OUTFIT_JSON = 30000\nMAX_OUTFIT_RESPONSE = 32 * 1024 * 1024\n\n\ndef _outfit_items(raw):\n    if len(raw) > MAX_OUTFIT_JSON:\n        raise HTTPException(413, \'Outfit evidence is too large\')\n    try:\n        value = json.loads(raw)\n    except (json.JSONDecodeError, TypeError) as exc:\n        raise HTTPException(400, \'Invalid outfit evidence\') from exc\n    if not isinstance(value, list) or not 1 <= len(value) <= MAX_OUTFIT_ITEMS:\n        raise HTTPException(400, \'Provide between one and five outfit items\')\n    clean, seen = [], set()\n    for item in value:\n        if not isinstance(item, dict) or not isinstance(item.get(\'index\'), int) or item[\'index\'] in seen:\n            raise HTTPException(400, \'Outfit item indexes must be unique integers\')\n        seen.add(item[\'index\'])\n        candidate = {\n            \'index\': item[\'index\'],\n            \'boundingBox\': item.get(\'boundingBox\'),\n            \'parserClass\': str(item.get(\'parserClass\') or \'\')[:40],\n            \'quality\': item.get(\'quality\', \'high\'),\n            \'seed\': item.get(\'seed\', 42),\n        }\n        category = item.get(\'category\')\n        manifest = item.get(\'manifest\')\n        if category is not None:\n            try:\n                category = normalize_category(category)\n                manifest = normalize_manifest(category, manifest)\n            except (ValueError, TypeError) as exc:\n                raise HTTPException(400, \'Invalid or mismatched outfit garment evidence\') from exc\n            if candidate[\'quality\'] not in (\'standard\', \'high\') or not isinstance(candidate[\'seed\'], int) or not 0 <= candidate[\'seed\'] < 2**32:\n                raise HTTPException(400, \'Unsupported outfit quality or seed\')\n            candidate.update(category=category, manifest=manifest)\n        clean.append(candidate)\n    return clean\n\n\n@app.post(\'/outfit\')\nasync def generate_outfit(image: UploadFile = File(...), items: str = Form(...),\n                          contract_version: int = Form(...)):\n    try:\n        if ENGINE is None or not getattr(ENGINE, \'is_warm\', False):\n            raise HTTPException(503, \'Engine is warming up; retry shortly.\', headers={\'Retry-After\':\'15\'})\n        if contract_version != CONTRACT_VERSION:\n            raise HTTPException(409, \'Update the website to the appearance v2 contract.\')\n        if image.content_type not in (\'image/png\', \'image/jpeg\', \'image/webp\'):\n            raise HTTPException(400, \'Invalid outfit image type\')\n        parsed_items = _outfit_items(items)\n        data = await image.read(MAX_UPLOAD_BYTES + 1)\n        if not data or len(data) > MAX_UPLOAD_BYTES:\n            raise HTTPException(413, \'Empty image or upload exceeds 20 MB\')\n        request_log(\'outfit_parsing_started\', item_count=len(parsed_items), input_bytes=len(data))\n        try:\n            cutouts, skipped, evidence = await asyncio.to_thread(parse_outfit, data, parsed_items)\n        except OutfitParserError as exc:\n            request_log(\'outfit_not_worn\', reason=str(exc))\n            raise HTTPException(409, str(exc), headers={\'X-Outfit-Pipeline-Version\':PARSER_VERSION}) from exc\n        by_index = {item[\'index\']: item for item in parsed_items}\n        results = []\n        for cutout in cutouts:\n            request = by_index[cutout.index]\n            result = {\n                \'index\': cutout.index,\n                \'sourcePng\': base64.b64encode(cutout.png).decode(\'ascii\'),\n                \'sourceWidth\': cutout.width,\n                \'sourceHeight\': cutout.height,\n                \'parserLabels\': list(cutout.labels),\n                \'sourcePixels\': cutout.pixel_count,\n                \'preservedOcclusionPixels\': cutout.preserved_occlusion_pixels,\n                \'repairedOcclusionPixels\': cutout.repaired_occlusion_pixels,\n                \'trimmedFootwearPixels\': cutout.trimmed_footwear_pixels,\n            }\n            if request.get(\'category\'):\n                try:\n                    rendered = await asyncio.to_thread(\n                        render_request, cutout.png, request[\'category\'], request[\'manifest\'],\n                        request[\'quality\'], request[\'seed\']\n                    )\n                    result.update(\n                        generatedPng=base64.b64encode(rendered.body).decode(\'ascii\'),\n                        category=request[\'category\'], seed=request[\'seed\']\n                    )\n                except Exception as exc:\n                    request_log(\'outfit_item_generation_failed\', index=cutout.index, error_type=type(exc).__name__)\n                    result[\'generationError\'] = str(exc.detail) if isinstance(exc, HTTPException) else \'3D generation failed for this item\'\n            results.append(result)\n        payload = {\'items\': results, \'skipped\': skipped, \'wornEvidence\': evidence,\n                   \'parserVersion\': PARSER_VERSION, \'contractVersion\': CONTRACT_VERSION}\n        encoded = json.dumps(payload, separators=(\',\', \':\')).encode()\n        if len(encoded) > MAX_OUTFIT_RESPONSE:\n            raise HTTPException(413, \'Prepared outfit response exceeds 32 MB\')\n        request_log(\'outfit_finished\', parsed=len(results), skipped=len(skipped), worn_evidence=evidence)\n        return Response(encoded, media_type=\'application/json\', headers={\n            \'X-Outfit-Pipeline-Version\': PARSER_VERSION,\n            \'X-Ghost-Contract-Version\': str(CONTRACT_VERSION),\n        })\n    finally:\n        await image.close()\n'
PROMPT_CONTRACT_CODE = '"""ClothMatics appearance contract v2. Pure Python, independently testable."""\nimport re\n\nCONTRACT_VERSION = 2\nCATEGORIES = {"shirt", "tshirt", "trackpants", "trousers", "cargo", "hoodie", "jacket", "dress", "shorts"}\nSHAPE_RULES = {\'trousers\': \'Lower garment only, waistband to both hems. Preserve the exact rise, centered crotch seam, fly, inseams, two separate leg tubes, leg width and both hem openings; never add a torso or turn trousers into a jumpsuit.\', \'trackpants\': \'Lower garment only, waistband to both hems. Preserve the exact elastic waist, drawcord, rise, crotch, two separate legs, pocket layout, leg silhouette and open or cuffed hems; never add an upper garment.\', \'cargo\': \'Lower garment only, waistband to both hems. Preserve rise, fly, crotch, two separate legs and every visible cargo/hip/rear pocket with its placement and flap; never add a torso.\', \'shorts\': \'Lower garment only, waistband to both short hems. Preserve rise, fly or drawcord, centered crotch, two separate leg openings, pocket layout and exact inseam length; never lengthen into trousers or add a torso.\', \'top\': \'Preserve the exact observed top neckline, armholes, straps, sleeves and hem; do not add shirt collars or plackets.\', \'knitwear\': \'Preserve the knit pattern, ribbing, neckline, sleeve or sleeveless construction and hem.\', \'robe\': \'Preserve the visible long flowing silhouette, wrap or front opening, belt, panels and coverage.\', \'draped\': \'Preserve the photographed wrapped fabric, folds, borders and coverage; do not stitch it into trousers or invent hidden drape.\', \'clothing_set\': \'Preserve exactly the visible separate garment pieces, lengths and layering; never fuse pieces or invent missing garments.\', \'sleepwear\': \'Preserve exactly the photographed nightwear pieces, straps, closures, coverage and lengths.\', \'skirt\': \'Lower garment only. Preserve skirt flare, pleats, layers and hem; never split into trouser legs.\', \'leggings\': \'Lower garment only, waistband to both hems. Preserve close-fitting stretch construction.\', \'saree\': \'Preserve the visible saree drape, pleats, pallu and border; do not turn draped cloth into a stitched dress or invent hidden pieces.\', \'lehenga\': \'Preserve the visible flared skirt, layers, border and any photographed set pieces; do not invent missing pieces.\', \'kurta\': \'Preserve tunic length, side slits, neckline and embroidery; never shorten to a western shirt.\', \'sherwani\': \'Preserve long coat panels, collar, closures and embroidery; do not shorten or add unseen trousers.\', \'traditional_set\': \'Preserve exactly the photographed set pieces, their separate layers, drape and borders; never fuse or add pieces.\', \'jumpsuit\': \'Keep the continuous one-piece bodice and divided legs, waistband and closures.\', \'romper\': \'Keep the continuous one-piece bodice and short divided legs; preserve inseam length.\', \'blouse\': \'Preserve the observed blouse cut, neckline, sleeves, ties and hem; never add a standard shirt placket.\', \'cardigan\': \'Preserve knit texture, opening, closures, neckline and length.\', \'swimwear\': \'Preserve the exact visible one-piece or separate-piece construction, straps and coverage.\', \'innerwear\': \'Preserve the photographed garment pieces, straps, cups, seams, elastic and coverage.\', \'scarf\': \'Preserve draped or folded fabric, length, borders and fringe; do not add a torso garment.\'}\nCATEGORIES.update(SHAPE_RULES)\nLOWER_CATEGORIES = {"trousers", "trackpants", "cargo", "shorts", "skirt", "leggings"}\nALIASES = {"t-shirt":"tshirt", "t_shirt":"tshirt", "tee":"tshirt", "jeans":"trousers", "denim":"trousers", "pants":"trousers", "chinos":"trousers", "joggers":"trackpants", "sweatpants":"trackpants", "blazer":"jacket", "coat":"jacket", "polo":"shirt", "sweatshirt":"tshirt"}\nFIELDS = [\n    ("colorAndFinish", "Photographed colors", 400, 70),\n    ("surfaceTextureAndWeave", "Fabric texture", 300, 48),\n    ("waistbandAndRise", "Waistband and rise", 200, 28),\n    ("flyAndClosure", "Fly and closure", 200, 24),\n    ("crotchAndInseam", "Crotch and inseam", 220, 30),\n    ("legSilhouette", "Leg silhouette", 200, 28),\n    ("hemAndCuffs", "Both hems or cuffs", 180, 24),\n    ("pocketLayout", "Pocket layout", 220, 28),\n    ("externalCompartments", "Pockets", 240, 36),\n    ("hardwareAndClosures", "Fasteners", 220, 36),\n    ("necklineOrWaistband", "Neckline or waistband", 200, 32),\n    ("garmentLengthAndHem", "Length and hem", 200, 32),\n    ("graphicsOrText", "Graphics and lettering", 300, 48),\n    ("fit", "Observed fit", 80, 16),\n    ("sleeveType", "Sleeves", 80, 16),\n]\n\ndef normalize_category(value):\n    raw = re.sub(r"\\s+", "_", str(value or "").strip().lower())\n    category = ALIASES.get(raw, raw)\n    if category not in CATEGORIES:\n        raise ValueError("Unsupported garment category; no generic shirt fallback")\n    return category\n\ndef category_dimensions(category, size, quality):\n    """Keep the proven portrait profile for tops; frame lower garments by source shape."""\n    category = normalize_category(category)\n    if quality not in ("high", "portrait", "standard") or category not in LOWER_CATEGORIES:\n        return (768, 1024) if quality in ("high", "portrait", "standard") else ((1024, 768) if quality == "wide" else (576, 768))\n    source_width, source_height = size\n    ratio = source_width / max(1, source_height)\n    if category == "shorts":\n        ratio = min(1.15, max(.78, ratio))\n        height = 896\n    else:\n        ratio = min(.82, max(.60, ratio))\n        height = 1024\n    width = max(16, round((height * ratio) / 16) * 16)\n    return width, height\n\ndef normalize_manifest(category, manifest):\n    category = normalize_category(category)\n    if not isinstance(manifest, dict) or normalize_category(manifest.get("category")) != category:\n        raise ValueError("Manifest category must match the requested garment")\n    result = {"category": category}\n    for key, _, limit, _ in FIELDS:\n        value = manifest.get(key, "")\n        if not isinstance(value, str):\n            raise ValueError("Manifest fields must be text")\n        result[key] = re.sub(r"[<>\\x00-\\x1f]", " ", value).strip()[:limit]\n    if not result["colorAndFinish"] or not result["surfaceTextureAndWeave"]:\n        raise ValueError("Observed color and fabric texture are required")\n    if category in LOWER_CATEGORIES:\n        result["sleeveType"] = ""\n    else:\n        for key in ("waistbandAndRise", "flyAndClosure", "crotchAndInseam", "legSilhouette", "hemAndCuffs", "pocketLayout"):\n            result[key] = ""\n    palette = manifest.get(\'palette\', [])\n    if not isinstance(palette, list) or len(palette) > 12:\n        raise ValueError(\'Invalid garment palette\')\n    result[\'palette\'] = []\n    for color in palette:\n        if not isinstance(color, dict) or color.get(\'role\') not in (\'base\',\'secondary\',\'print\',\'trim\',\'hardware\',\'wash\',\'embroidery\',\'panel\') or not re.fullmatch(r\'#[a-fA-F0-9]{6}\', str(color.get(\'hex\', \'\'))):\n            raise ValueError(\'Invalid color sample\')\n        region = re.sub(r\'[^a-zA-Z0-9 ,/-]\', \' \', str(color.get(\'region\', \'\'))).strip()[:48]\n        result[\'palette\'].append({\'role\':color[\'role\'], \'hex\':color[\'hex\'].upper(), \'region\':region})\n    return result\n\ndef invariant_prompt(category):\n    category = normalize_category(category)\n    lower = category in LOWER_CATEGORIES\n    shape = SHAPE_RULES.get(category) or ("Lower garment only, waistband to leg hems. No upper garment or jumpsuit. " if lower else "Keep the reference garment\'s observed sleeves, neckline and hem. ")\n    volume = ("visible inner waistband depth, natural seat and crotch volume, two separate leg tubes, sidewall thickness, fold gradients, contact shadows and subtle product-camera perspective" if lower else "inner edge depth at the collar or waistband, natural shoulder or seat shape, sidewall thickness, fold gradients, contact shadows and subtle product-camera perspective")\n    jacket = ("Jacket fidelity is strict: preserve the exact collar and front closure, pocket count and placement, sleeve marks and trim. Keep every left/right detail on the same viewer side; never mirror the reference. Do not add pockets, snaps, panels or logos that are not visibly present. The collar must be a truly empty garment opening with background or natural dark inner-fabric depth visible through it; never place a white, grey or skin-toned neck, chest or mannequin surface inside. " if category == "jacket" else "")\n    return (\n        f"Create a clean studio ghost-mannequin product render of the SAME single {category}; no visible mannequin or human body. Use a completely invisible, anatomically neutral garment support. "\n        f"Keep the support hidden while giving the clothing believable three-dimensional volume: {volume}. "\n        "Never make a flat front cutout, technical drawing or 2D icon. No visible mannequin, person, skin, head, neck cylinder, torso, limbs, stand or hanger; only garment and white background may be visible. "\n        + jacket + shape +\n        "Reference garment pixels override text color names. Preserve photographed hue, saturation, brightness, white balance, "\n        "texture, cut, pockets, fasteners and lettering. No recoloring, redesign or invented hidden details. "\n    )\n\ndef pack_prompt(category, manifest, tokenizer, max_tokens=460):\n    """Reserve space for the chat template; never strip or truncate invariants.\n\n    Distribute the remaining budget across individual evidence fields. Shrink\n    field values (not a blind tail slice) so every observed property gets space.\n    """\n    manifest = normalize_manifest(category, manifest)\n    head = invariant_prompt(category)\n    palette = manifest[\'palette\']\n    if palette:\n        head += \' Preserve each sampled region separately; never average colors or neutralize warm stripes: \' + \'; \'.join(f"{c[\'role\']} {c[\'region\']} {c[\'hex\']}" for c in palette) + \'. \'\n    encode = lambda text: tokenizer.encode(text, add_special_tokens=False)\n    if len(encode(head)) > max_tokens - 80:\n        raise ValueError("Color evidence exceeds model context; shorten region labels and retry analysis. No palette was silently truncated.")\n    sections, truncated = [], []\n    for key, label, _, quota in FIELDS:\n        value = manifest[key]\n        if not value:\n            continue\n        ids = encode(value)\n        sections.append([key, label, ids, min(len(ids), quota)])\n        if len(ids) > quota:\n            truncated.append(key)\n    def assemble():\n        return head + " ".join(f"{label}: {tokenizer.decode(ids[:count], skip_special_tokens=True)}." for _, label, ids, count in sections)\n    while len(encode(assemble())) > max_tokens:\n        candidates = [s for s in sections if s[3] > 8]\n        if not candidates:\n            raise ValueError("Manifest cannot fit safely in the model context")\n        section = max(candidates, key=lambda s:s[3])\n        section[3] -= 1\n        if section[0] not in truncated:\n            truncated.append(section[0])\n    prompt = assemble()\n    return prompt, {"prompt_tokens":len(encode(prompt)), "truncated_fields":truncated, "contract_version":CONTRACT_VERSION}\n'
OUTFIT_PARSER_CODE = '"""Pixel-level worn-outfit extraction for the optional /outfit route.\n\nThis module is deliberately isolated from ghost_server.py.  The established\n/generate renderer never imports or loads the parser unless /outfit is used.\n"""\nfrom __future__ import annotations\n\nimport io\nimport os\nimport threading\nfrom dataclasses import dataclass\nfrom typing import Iterable\n\nimport cv2\nimport numpy as np\nfrom PIL import Image, ImageOps, ImageCms, UnidentifiedImageError\n\nMODEL_ID = "mattmdjaga/segformer_b2_clothes"\nMODEL_REVISION = "584abc1e1d260e23c0fc627c5217a09b2b461046"\nPARSER_VERSION = "3"\nMAX_PIXELS = 24_000_000\nSRGB_PROFILE = ImageCms.ImageCmsProfile(ImageCms.createProfile("sRGB"))\nSRGB_ICC = SRGB_PROFILE.tobytes()\n\n# Model labels: 0 background, 1 hat, 2 hair, 3 sunglasses, 4 upper-clothes,\n# 5 skirt, 6 pants, 7 dress, 8 belt, 9/10 shoes, 11 face, 12/13 legs,\n# 14/15 arms, 16 bag, 17 scarf.\nBODY_LABELS = frozenset((2, 11, 12, 13, 14, 15))\nGARMENT_LABELS = frozenset((4, 5, 6, 7))\nFOOTWEAR_LABELS = frozenset((9, 10))\n\n_MODEL = None\n_PROCESSOR = None\n_LOAD_LOCK = threading.Lock()\n\n\nclass OutfitParserError(ValueError):\n    pass\n\n\n@dataclass\nclass ParsedCutout:\n    index: int\n    png: bytes\n    width: int\n    height: int\n    pixel_count: int\n    labels: tuple[int, ...]\n    preserved_occlusion_pixels: int = 0\n    repaired_occlusion_pixels: int = 0\n    trimmed_footwear_pixels: int = 0\n\n\ndef _load_parser():\n    global _MODEL, _PROCESSOR\n    if _MODEL is not None:\n        return _PROCESSOR, _MODEL\n    with _LOAD_LOCK:\n        if _MODEL is None:\n            import torch\n            from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor\n\n            device = os.environ.get("CLOTHMATICS_PARSER_DEVICE", "cpu").strip().lower()\n            if device != "cpu":\n                raise RuntimeError("The outfit parser is pinned to CPU so FLUX GPU behavior remains unchanged")\n            _PROCESSOR = SegformerImageProcessor.from_pretrained(\n                MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False\n            )\n            _MODEL = SegformerForSemanticSegmentation.from_pretrained(\n                MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False\n            ).to("cpu").eval()\n    return _PROCESSOR, _MODEL\n\n\ndef decode_photo(data: bytes) -> Image.Image:\n    if not data:\n        raise OutfitParserError("The outfit photo is empty")\n    try:\n        with Image.open(io.BytesIO(data)) as opened:\n            if getattr(opened, "n_frames", 1) != 1:\n                raise OutfitParserError("Upload one still outfit photo")\n            if opened.width * opened.height > MAX_PIXELS:\n                raise OutfitParserError("The outfit photo exceeds 24M pixels")\n            opened.load()\n            icc = opened.info.get("icc_profile")\n            oriented = ImageOps.exif_transpose(opened)\n            if icc:\n                try:\n                    source_profile = ImageCms.ImageCmsProfile(io.BytesIO(icc))\n                    base = oriented if oriented.mode in ("RGB", "CMYK", "L", "LAB") else oriented.convert("RGB")\n                    return ImageCms.profileToProfile(base, source_profile, SRGB_PROFILE, outputMode="RGB")\n                except (ImageCms.PyCMSError, OSError, ValueError) as exc:\n                    raise OutfitParserError("Embedded color profile cannot be converted") from exc\n            return oriented.convert("RGB")\n    except OutfitParserError:\n        raise\n    except (UnidentifiedImageError, OSError, ValueError) as exc:\n        raise OutfitParserError("Unsupported outfit photo") from exc\n\n\ndef predict_labels(photo: Image.Image) -> np.ndarray:\n    import torch\n\n    processor, model = _load_parser()\n    inputs = processor(images=photo, return_tensors="pt")\n    with torch.inference_mode():\n        outputs = model(**inputs)\n    target = [(photo.height, photo.width)]\n    # Transformers 5.x expects SemanticSegmenterOutput here and reads\n    # outputs.logits internally. Passing the tensor itself raises AttributeError.\n    labels = processor.post_process_semantic_segmentation(outputs, target_sizes=target)[0]\n    return labels.detach().cpu().numpy().astype(np.uint8)\n\n\ndef worn_photo_evidence(labels: np.ndarray) -> dict:\n    total = max(1, labels.size)\n    body_pixels = int(np.isin(labels, tuple(BODY_LABELS)).sum())\n    garment_pixels = int(np.isin(labels, tuple(GARMENT_LABELS)).sum())\n    face = int((labels == 11).sum())\n    limbs = int(np.isin(labels, (12, 13, 14, 15)).sum())\n    # Require both clothing and visible human evidence. This keeps flat-lay and\n    # hanging garment photos on the established /generate path.\n    worn = garment_pixels / total >= 0.004 and body_pixels >= 64 and (\n        face > 0 or limbs > 0\n    )\n    return {\n        "worn": bool(worn),\n        "body_fraction": round(body_pixels / total, 5),\n        "garment_fraction": round(garment_pixels / total, 5),\n    }\n\n\ndef _box(value, width: int, height: int) -> tuple[int, int, int, int]:\n    if not isinstance(value, (list, tuple)) or len(value) != 4:\n        raise OutfitParserError("Each outfit item needs a four-value bounding box")\n    try:\n        y1, x1, y2, x2 = [float(part) for part in value]\n    except (TypeError, ValueError) as exc:\n        raise OutfitParserError("Invalid outfit item bounding box") from exc\n    # The website contract always uses normalized 0..1000 coordinates.\n    if min(y1, x1, y2, x2) < 0 or max(y1, x1, y2, x2) > 1000:\n        raise OutfitParserError("Outfit item bounding box must use 0..1000 coordinates")\n    y1, y2 = y1 * height / 1000, y2 * height / 1000\n    x1, x2 = x1 * width / 1000, x2 * width / 1000\n    if y2 <= y1 or x2 <= x1:\n        raise OutfitParserError("Invalid outfit item bounding box")\n    pad_x = max(3, int((x2 - x1) * 0.08))\n    pad_y = max(3, int((y2 - y1) * 0.08))\n    return (\n        max(0, int(x1) - pad_x), max(0, int(y1) - pad_y),\n        min(width, int(np.ceil(x2)) + pad_x), min(height, int(np.ceil(y2)) + pad_y),\n    )\n\n\ndef labels_for(kind: str) -> tuple[int, ...]:\n    key = str(kind or "").lower()\n    if key in ("shirt", "tshirt", "hoodie", "jacket", "blouse", "cardigan", "knitwear", "top", "sherwani", "kurta"):\n        return (4,)\n    if key == "skirt":\n        return (5,)\n    if key in ("trousers", "trackpants", "cargo", "shorts", "leggings"):\n        return (6,)\n    if key in ("dress", "jumpsuit", "romper", "saree", "lehenga", "traditional_set", "robe", "draped", "sleepwear", "clothing_set", "swimwear", "innerwear"):\n        return (7, 4, 5, 6)\n    if key == "scarf":\n        return (17,)\n    if key == "footwear":\n        return (9, 10)\n    raise OutfitParserError("Unsupported outfit item type")\n\n\ndef _clean_mask(mask: np.ndarray) -> np.ndarray:\n    binary = mask.astype(np.uint8)\n    kernel = np.ones((3, 3), np.uint8)\n    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=1)\n    count, component, stats, _ = cv2.connectedComponentsWithStats(binary, 8)\n    if count <= 1:\n        return binary.astype(bool)\n    minimum = max(32, int(binary.size * 0.00012))\n    keep = np.zeros_like(binary)\n    for label in range(1, count):\n        if stats[label, cv2.CC_STAT_AREA] >= minimum:\n            keep[component == label] = 1\n    return keep.astype(bool)\n\n\ndef _convex_envelope(mask: np.ndarray) -> np.ndarray:\n    """Return a conservative silhouette envelope around observed garment pixels."""\n    points = np.column_stack(np.where(mask)[::-1]).astype(np.int32)\n    if len(points) < 3:\n        return mask.copy()\n    hull = cv2.convexHull(points.reshape(-1, 1, 2))\n    envelope = np.zeros(mask.shape, dtype=np.uint8)\n    cv2.fillConvexPoly(envelope, hull, 1)\n    envelope = cv2.dilate(envelope, np.ones((3, 3), np.uint8), iterations=1)\n    return envelope.astype(bool)\n\n\ndef _preserve_occlusions(mask: np.ndarray, labels: np.ndarray, region: np.ndarray,\n                         parser_class: str) -> tuple[np.ndarray, np.ndarray]:\n    """Keep photographed occluder pixels that sit inside the garment silhouette.\n\n    This intentionally preserves source pixels (for example a hand resting over\n    trousers) instead of cutting holes into the garment.  It never synthesizes\n    hidden textile pixels; the downstream garment renderer removes the person.\n    """\n    if not mask.any():\n        return mask, np.zeros(mask.shape, dtype=bool)\n    kind = str(parser_class or "").lower()\n    envelope = _convex_envelope(mask) & region\n    if kind in ("trousers", "trackpants", "cargo", "shorts", "leggings", "skirt"):\n        ys = np.where(mask)[0]\n        top, bottom = int(ys.min()), int(ys.max()) + 1\n        upper_limit = top + max(1, int((bottom - top) * 0.46))\n        upper_band = np.zeros(mask.shape, dtype=bool)\n        upper_band[top:upper_limit] = True\n        # Hands/arms, an overlapping shirt hem and a belt commonly cover the\n        # waistband/hip area.  Preserve those photographed pixels only within\n        # the inferred lower-garment envelope and its upper section.\n        candidates = np.isin(labels, (4, 8, 14, 15)) & upper_band\n    elif kind in ("shirt", "tshirt", "hoodie", "jacket", "blouse", "cardigan",\n                  "knitwear", "top", "sherwani", "kurta"):\n        # Keep arms/hands where they cross the shirt body.  Face/neck pixels are\n        # deliberately excluded so the collar opening remains transparent.\n        candidates = np.isin(labels, (14, 15))\n    elif kind in ("dress", "jumpsuit", "romper", "saree", "lehenga",\n                  "traditional_set", "robe", "draped", "sleepwear",\n                  "clothing_set", "swimwear", "innerwear"):\n        candidates = np.isin(labels, (14, 15))\n    else:\n        return mask, np.zeros(mask.shape, dtype=bool)\n    additions = candidates & envelope & region & ~mask\n    return mask | additions, additions\n\n\ndef _repair_occlusion_pixels(source_rgb: np.ndarray, clean_garment: np.ndarray,\n                             occlusions: np.ndarray) -> tuple[np.ndarray, int]:\n    """Replace small verified occluders with nearest observed garment pixels.\n\n    The fill is restricted to arm/hand or overlapping-clothing labels already\n    accepted by ``_preserve_occlusions``.  It copies source garment pixels and\n    does not sample skin or background.  Very large hidden regions are retained\n    as photographed because their unseen texture cannot be recovered reliably.\n    """\n    count = int(occlusions.sum())\n    clean_count = int(clean_garment.sum())\n    if not count or clean_count < 64 or count > clean_count * 0.22:\n        return source_rgb, 0\n    ys, xs = np.where(clean_garment | occlusions)\n    pad = 2\n    top, bottom = max(0, int(ys.min()) - pad), min(source_rgb.shape[0], int(ys.max()) + pad + 1)\n    left, right = max(0, int(xs.min()) - pad), min(source_rgb.shape[1], int(xs.max()) + pad + 1)\n    clean = clean_garment[top:bottom, left:right]\n    target = occlusions[top:bottom, left:right]\n    # distanceTransformWithLabels assigns every non-clean pixel the label of\n    # its nearest clean garment pixel.  The lookup table is built only from\n    # observed garment pixels, so hands cannot leak back into the repair.\n    _, nearest = cv2.distanceTransformWithLabels(\n        (~clean).astype(np.uint8), cv2.DIST_L2, 5,\n        labelType=cv2.DIST_LABEL_PIXEL\n    )\n    clean_y, clean_x = np.where(clean)\n    if not len(clean_x):\n        return source_rgb, 0\n    repaired = source_rgb.copy()\n    local = repaired[top:bottom, left:right]\n    target_y, target_x = np.where(target)\n    nearest_index = nearest[target_y, target_x] - 1\n    valid = (nearest_index >= 0) & (nearest_index < len(clean_x))\n    if not valid.any():\n        return source_rgb, 0\n    local[target_y[valid], target_x[valid]] = local[\n        clean_y[nearest_index[valid]], clean_x[nearest_index[valid]]\n    ]\n    return repaired, int(valid.sum())\n\n\ndef _trim_footwear_stems(mask: np.ndarray) -> tuple[np.ndarray, int]:\n    """Remove narrow sock/ankle stems above each otherwise intact shoe mask."""\n    binary = mask.astype(np.uint8)\n    count, components, stats, _ = cv2.connectedComponentsWithStats(binary, 8)\n    result = binary.copy()\n    trimmed = 0\n    for label in range(1, count):\n        x = stats[label, cv2.CC_STAT_LEFT]\n        y = stats[label, cv2.CC_STAT_TOP]\n        width = stats[label, cv2.CC_STAT_WIDTH]\n        height = stats[label, cv2.CC_STAT_HEIGHT]\n        if width < 8 or height < 12:\n            continue\n        component = components[y:y + height, x:x + width] == label\n        row_widths = component.sum(axis=1)\n        maximum = int(row_widths.max())\n        threshold = max(4, int(maximum * 0.34))\n        sustained = np.convolve((row_widths >= threshold).astype(np.uint8),\n                                np.ones(3, dtype=np.uint8), mode="same")\n        candidates = np.where(sustained >= 3)[0]\n        if not len(candidates):\n            continue\n        body_start = max(0, int(candidates[0]) - 1)\n        minimum_stem = max(3, int(height * 0.07))\n        maximum_trim = int(height * 0.34)\n        if body_start < minimum_stem or body_start > maximum_trim:\n            continue\n        stem_width = float(row_widths[:body_start].mean()) if body_start else maximum\n        if stem_width > maximum * 0.48:\n            continue\n        stem = component[:body_start]\n        removed = int(stem.sum())\n        if removed:\n            view = result[y:y + body_start, x:x + width]\n            view[stem] = 0\n            trimmed += removed\n    return result.astype(bool), trimmed\n\n\ndef cutout_for(photo: Image.Image, labels: np.ndarray, item: dict) -> ParsedCutout:\n    index = int(item.get("index", -1))\n    selected = labels_for(item.get("parserClass") or item.get("category"))\n    x1, y1, x2, y2 = _box(item.get("boundingBox"), photo.width, photo.height)\n    region = np.zeros(labels.shape, dtype=bool)\n    region[y1:y2, x1:x2] = True\n    parser_class = str(item.get("parserClass") or item.get("category") or "").lower()\n    mask = _clean_mask(np.isin(labels, selected) & region)\n    ys, xs = np.where(mask)\n    minimum = max(48, int((x2 - x1) * (y2 - y1) * 0.005))\n    if len(xs) < minimum and item.get("parserClass") != "footwear":\n        # A fashion parser can label an unusual top as dress or loose shorts as\n        # skirt. Stay on semantic clothing pixels, choose the dominant class\n        # inside this item\'s Gemini box, and never fall back to the rectangle.\n        counts = [(int(((labels == label) & region).sum()), label) for label in GARMENT_LABELS]\n        count, fallback = max(counts, default=(0, 0))\n        if count >= minimum:\n            selected = (fallback,)\n            mask = _clean_mask((labels == fallback) & region)\n            ys, xs = np.where(mask)\n    if len(xs) < minimum:\n        raise OutfitParserError("The semantic parser found too few garment pixels inside this item box")\n    preserved_occlusion_pixels = 0\n    repaired_occlusion_pixels = 0\n    trimmed_footwear_pixels = 0\n    source_rgb = np.asarray(photo).copy()\n    if parser_class == "footwear":\n        mask, trimmed_footwear_pixels = _trim_footwear_stems(mask)\n    else:\n        clean_garment = mask.copy()\n        mask, occlusions = _preserve_occlusions(\n            mask, labels, region, parser_class\n        )\n        source_rgb, repaired_occlusion_pixels = _repair_occlusion_pixels(\n            source_rgb, clean_garment, occlusions\n        )\n        preserved_occlusion_pixels = int(occlusions.sum()) - repaired_occlusion_pixels\n    ys, xs = np.where(mask)\n    left, right = int(xs.min()), int(xs.max()) + 1\n    top, bottom = int(ys.min()), int(ys.max()) + 1\n    margin = max(2, int(max(right - left, bottom - top) * 0.015))\n    left, top = max(0, left - margin), max(0, top - margin)\n    right, bottom = min(photo.width, right + margin), min(photo.height, bottom + margin)\n    alpha = (mask.astype(np.uint8) * 255)\n    # A one-pixel edge softening avoids a jagged preview without adding any\n    # pixels that were not present in the source photograph.\n    alpha = cv2.GaussianBlur(alpha, (3, 3), 0.45)\n    rgba = np.dstack((source_rgb, alpha))[top:bottom, left:right]\n    output = Image.fromarray(rgba, "RGBA")\n    if max(output.size) > 1200:\n        output.thumbnail((1200, 1200), Image.Resampling.LANCZOS)\n    buffer = io.BytesIO()\n    output.save(buffer, format="PNG", optimize=True, icc_profile=SRGB_ICC)\n    return ParsedCutout(\n        index, buffer.getvalue(), output.width, output.height, int(mask.sum()), selected,\n        preserved_occlusion_pixels, repaired_occlusion_pixels, trimmed_footwear_pixels\n    )\n\n\ndef parse_outfit(data: bytes, items: Iterable[dict]) -> tuple[list[ParsedCutout], list[dict], dict]:\n    photo = decode_photo(data)\n    labels = predict_labels(photo)\n    evidence = worn_photo_evidence(labels)\n    if not evidence["worn"]:\n        raise OutfitParserError("The parser did not find reliable worn-person evidence")\n    parsed, skipped = [], []\n    for item in items:\n        try:\n            parsed.append(cutout_for(photo, labels, item))\n        except (OutfitParserError, TypeError, ValueError) as exc:\n            skipped.append({"index": int(item.get("index", -1)), "message": str(exc)})\n    return parsed, skipped, evidence\n'

(PROJECT / 'clothmatics_api.py').write_text(WARM_SERVER_CODE)
(PROJECT / 'prompt_contract.py').write_text(PROMPT_CONTRACT_CODE)
(PROJECT / 'outfit_parser.py').write_text(OUTFIT_PARSER_CODE)
print("✅ Safe Baseline FastAPI server script written.")

# ------------------------------------------------------------------------------
# STEP 5: Start FastAPI Server & Wait for Warm Model Loading
# ------------------------------------------------------------------------------
with socket.socket() as sock:
    sock.bind(('127.0.0.1', 0))
    API_PORT = sock.getsockname()[1]
API_URL = f'http://127.0.0.1:{API_PORT}'

api_env = dict(os.environ)
api_env['CLOTHMATICS_PROJECT'] = str(PROJECT.resolve())
api_env['PYTHONUNBUFFERED'] = '1'
API_LOG = PROJECT / 'api-server.log'
API_LOG_FILE = open(API_LOG, 'w', encoding='utf-8', buffering=1)

print(f"\nStarting API Server on port {API_PORT}...")
print("⏳ Pre-loading models into GPU VRAM & executing preflight warm-up pass...")
API_PROCESS = subprocess.Popen([
    PYTHON, '-m', 'uvicorn', 'clothmatics_api:app',
    '--app-dir', str(PROJECT), '--host', '127.0.0.1', '--port', str(API_PORT),
    '--workers', '1', '--no-access-log', '--log-level', 'warning'
], cwd=PROJECT, env=api_env, stdout=API_LOG_FILE, stderr=subprocess.STDOUT)

model_ready = False
MAX_WAIT_SECONDS = 600
poll_interval = 2
max_attempts = MAX_WAIT_SECONDS // poll_interval
log_pos = 0

for attempt in range(max_attempts):
    time.sleep(poll_interval)
    elapsed = (attempt + 1) * poll_interval

    if API_LOG.exists():
        try:
            with API_LOG.open('r', encoding='utf-8', errors='replace') as lf:
                lf.seek(log_pos)
                new_chunk = lf.read()
                log_pos = lf.tell()
                if new_chunk:
                    for raw_line in new_chunk.splitlines():
                        clean_line = raw_line.strip()
                        if clean_line:
                            print(f"  [vram-engine] {clean_line}", flush=True)
        except Exception:
            pass

    if API_PROCESS.poll() is not None:
        log_content = API_LOG.read_text(encoding='utf-8', errors='replace') if API_LOG.exists() else "No log file found."
        raise RuntimeError("API process died unexpectedly:\n" + log_content[-3000:])

    status = "connecting"
    server_error = None
    try:
        req = urllib.request.Request(API_URL + '/health')
        with urllib.request.urlopen(req, timeout=3) as resp:
            data = json.loads(resp.read().decode())
            if data.get('error'):
                server_error = data['error']
            elif data.get('ready') and data.get('ghost_contract_version') == 2:
                model_ready = True
                break
            else:
                status = data.get('status', 'warming_up')
    except urllib.error.URLError:
        status = "connecting_to_server"
    except Exception as e:
        status = str(e)

    if server_error:
        print(f"\n❌ FATAL: Engine initialization failed in VRAM:\n{server_error}\n", flush=True)
        if API_LOG.exists():
            print("--- API Server Log Tail ---")
            print(API_LOG.read_text(encoding='utf-8', errors='replace')[-3000:])
        raise RuntimeError(f"Engine failed to load in VRAM:\n{server_error}")

    if attempt % 5 == 0 and attempt > 0:
        print(f"⏳ Waiting for model readiness in VRAM... ({elapsed}s/{MAX_WAIT_SECONDS}s) [Status: {status}]", flush=True)

if not model_ready:
    log_content = API_LOG.read_text() if API_LOG.exists() else "No log file found."
    raise RuntimeError(f"Model loading timed out after {MAX_WAIT_SECONDS}s. API Server Log:\n{log_content[-4000:]}")

print("🎉 MODEL IS WARM & PERMANENTLY LOADED IN VRAM!")

# ------------------------------------------------------------------------------
# STEP 6: Cloudflare Tunnel & Worker Auto-Registration
# ------------------------------------------------------------------------------
CLOUDFLARED = PROJECT / 'cloudflared'
if not CLOUDFLARED.exists():
    print("Downloading Cloudflare Tunnel binary...")
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', str(CLOUDFLARED)
    ], check=True)
    subprocess.run(['chmod', '+x', str(CLOUDFLARED)], check=True)

TUNNEL_LOG = PROJECT / 'tunnel.log'
TUNNEL_LOG_FILE = open(TUNNEL_LOG, 'w', encoding='utf-8', buffering=1)
TUNNEL_PROCESS = subprocess.Popen([
    str(CLOUDFLARED), 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{API_PORT}'
], stdout=TUNNEL_LOG_FILE, stderr=subprocess.STDOUT)

PUBLIC_API_URL = None
print("Connecting Cloudflare Tunnel...")
for _ in range(40):
    time.sleep(0.5)
    if TUNNEL_LOG.exists():
        text = TUNNEL_LOG.read_text(encoding='utf-8', errors='replace')
        matches = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', text)
        if matches:
            PUBLIC_API_URL = matches[0]
            break

if not PUBLIC_API_URL:
    raise RuntimeError("Failed to obtain Cloudflare tunnel URL within 20s. Check tunnel.log.")

# Register active tunnel with permanent Cloudflare Worker via POST with X-Sync-Token header
WORKER_SYNC_URL = "https://clothmatics-ghost.chiragsharma376.workers.dev/set-target"

try:
    import urllib.parse
    parsed_tunnel = urllib.parse.urlparse(PUBLIC_API_URL)
    if parsed_tunnel.scheme == "https" and parsed_tunnel.netloc:
        origin_tunnel = f"https://{parsed_tunnel.netloc}"
        sync_payload = json.dumps({"url": origin_tunnel}).encode("utf-8")
        req = urllib.request.Request(
            WORKER_SYNC_URL,
            data=sync_payload,
            headers={
                'Content-Type': 'application/json',
                'User-Agent': 'ClothMatics-Kaggle-Node/9.1.0',
                'X-Sync-Token': SYNC_TOKEN
            },
            method='POST'
        )
        with urllib.request.urlopen(req, timeout=10) as r:
            data = json.loads(r.read().decode())
            print(f"✅ WORKER LINKED: {data}")
    else:
        print(f"⚠️ Invalid tunnel URL format: {PUBLIC_API_URL}")
except Exception as e:
    for proc in [API_PROCESS, TUNNEL_PROCESS]:
        if proc.poll() is None:
            proc.terminate()
    raise RuntimeError('The new engine started but Worker registration failed. Check the matching Kaggle sync secret and Worker configuration.') from e

# ------------------------------------------------------------------------------
# FINISHED!
# ------------------------------------------------------------------------------
print("\n" + "="*80)
print("CLOTHMATICS v9.3.0 OUTFIT ROUTER IS LIVE")
print("="*80)
print(f"👉 PERMANENT WEBSITE ENDPOINT : https://clothmatics-ghost.chiragsharma376.workers.dev/generate")
print(f"👉 WORN OUTFIT ENDPOINT       : https://clothmatics-ghost.chiragsharma376.workers.dev/outfit")
print(f"👉 ACTIVE KAGGLE TUNNEL       : {PUBLIC_API_URL}")
print(f"⚡ COMPUTE PRECISION          : FP16 (Tensor Cores on Tesla T4) + FP32 Tiled VAE")
print(f"🛡️ PIPELINE MODE              : 3D Ghost Mannequin Safe Baseline (Non-destructive)")
print(f"GARMENT CONDITIONING         : category-specific, token-budgeted observed details")
print(f"COLOR HANDLING               : sRGB input; no global recoloring. Website visual comparison required.")
print(f"OUTFIT PARSER                : pinned SegFormer B2 on CPU; current FLUX renderer unchanged")
print(f"⏱️ INSTRUMENTATION            : Stage timings & quality status exposed in response headers")
print("="*80)
print("💡 Ready for generation requests via Cloudflare Worker proxy or active Kaggle tunnel.\n")

# ------------------------------------------------------------------------------
# STEP 7: Quiet listener & request-only diagnostics
# ------------------------------------------------------------------------------
print("📡 Ready. Idle output is quiet; generation requests print diagnostics (cell stays active [*]).")
print("   To stop the server, click the Stop button in Kaggle.\n")

last_api_pos = log_pos if 'log_pos' in globals() else (API_LOG.stat().st_size if API_LOG.exists() else 0)

try:
    while True:
        time.sleep(1)

        # 1. Process Health Checks
        if API_PROCESS.poll() is not None:
            tail = API_LOG.read_text(encoding='utf-8', errors='replace')[-3000:] if API_LOG.exists() else "No log file."
            print(f"\n❌ FATAL: API Server died unexpectedly (code {API_PROCESS.returncode}):\n{tail}", flush=True)
            break

        if TUNNEL_PROCESS.poll() is not None:
            tail = TUNNEL_LOG.read_text(encoding='utf-8', errors='replace')[-3000:] if TUNNEL_LOG.exists() else "No log file."
            print(f"\n❌ FATAL: Cloudflare Tunnel died unexpectedly (code {TUNNEL_PROCESS.returncode}):\n{tail}", flush=True)
            break

        # 2. Stream new API logs in real time (requests, processing, 200 OK)
        if API_LOG.exists():
            curr_size = API_LOG.stat().st_size
            if curr_size > last_api_pos:
                try:
                    with API_LOG.open('r', encoding='utf-8', errors='replace') as lf:
                        lf.seek(last_api_pos)
                        new_content = lf.read()
                        last_api_pos = lf.tell()
                        if new_content:
                            for raw_line in new_content.splitlines():
                                sline = raw_line.strip()
                                if sline:
                                    print(f"  {sline}", flush=True)
                except Exception:
                    pass

except KeyboardInterrupt:
    print("\n🛑 Stop requested by user (KeyboardInterrupt).")
finally:
    print("Shutting down API server and Cloudflare tunnel...")
    for proc in [API_PROCESS, TUNNEL_PROCESS]:
        if proc and proc.poll() is None:
            try:
                proc.terminate()
                proc.wait(timeout=3)
            except Exception:
                proc.kill()
    for fh_name in ['API_LOG_FILE', 'TUNNEL_LOG_FILE']:
        if fh_name in globals():
            try:
                globals()[fh_name].close()
            except Exception:
                pass
    print("👋 Clean shutdown complete. Kaggle resources released.")
